# Multi Agent : Autogen 실습 : StockAgent  
삼성전자와 SK하이닉스 주식 성과를 비교해서 투자 관련 참고 의견만 제공

### 구굴 Colab Drive 연결 및 OpenAI key 등록

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/aiffel/3.rag/4

/content/drive/MyDrive/aiffel/3.rag/4


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("/content/drive/MyDrive/aiffel/env_keys/.env")

print("OPENAI_API_KEY loaded:", os.getenv("OPENAI_API_KEY") is not None)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY loaded: True


### 주가 갖고오는 프로그램

In [ ]:
!pip install pykrx

In [ ]:
from pykrx import stock
import pandas as pd
import numpy as np

# def fetch_stock_history(ticker: str, period: str = "2y") -> pd.DataFrame:
#     # period를 간단히 날짜로 변환 (예: 2y → 약 2년 전)
#     end = pd.Timestamp.today().strftime("%Y%m%d")
#     start = (pd.Timestamp.today() - pd.DateOffset(years=2)).strftime("%Y%m%d")
#     df = stock.get_market_ohlcv(start, end, ticker)
#     return df

# data_1 = fetch_stock_history("005930")  # 삼성전자
# data_2 = fetch_stock_history("000660")  # sk 하이닉스

# # 컬럼에 티커 정보를 붙여서 MultiIndex로 만들고 합치기
# data_1_cols = data_1.add_prefix("005930_")
# data_2_cols = data_2.add_prefix("000660_")

# df = pd.concat([data_1_cols, data_2_cols], axis=1)

# print(df.head())

In [ ]:
def compute_metrics(df, price_col="Close", risk_free_rate=0.0, freq=252):
    """가격 시계열 df에서 수익률, 변동성, MDD, Sharpe 등을 계산."""
    # 1. 가격 → 일간 수익률 계산
    r = df[price_col].pct_change().dropna()

    # 2. 연간화 수익률, 변동성
    annual_return = r.mean() * freq
    annual_vol = r.std() * np.sqrt(freq)

    # 3. 최대 낙폭(Maximum Drawdown, MDD)
    cum_ret = (1 + r).cumprod()
    cum_max = cum_ret.cummax()
    drawdown = (cum_ret - cum_max) / cum_max
    max_drawdown = drawdown.min()

    # 4. Sharpe ratio (무위험수익률 0 가정)
    excess_return = annual_return - risk_free_rate
    sharpe = excess_return / annual_vol if annual_vol > 1e-8 else np.nan

    # 5. 기하 평균 수익률 (CAGR)
    cagr = cum_ret.iloc[-1] ** (freq / len(r)) - 1

    # 6. 결과 딕셔너리로 반환
    metrics = {
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "max_drawdown": max_drawdown,
        "sharpe_ratio": sharpe,
        "cagr": cagr,
        "n_days": len(r)
    }
    return pd.Series(metrics)


In [ ]:
# # 툴 역할 함수: df -> 두 종목 메트릭 계산
# def compute_stock_metrics(df: pd.DataFrame):
#     m_005930 = compute_metrics(df.rename(columns={"005930_종가": "Close"}))
#     m_000660 = compute_metrics(df.rename(columns={"000660_종가": "Close"}))
#     return {
#         "005930": m_005930.to_dict(),
#         "000660": m_000660.to_dict(),
#     }

In [ ]:
# # 예: 두 종목의 일간 수익률 계산
# returns = df.pct_change().dropna()

# # 예: 각 종목 별로 compute_metrics 적용
# metrics_005930 = compute_metrics(df.rename(columns={"005930_종가": "Close"}))
# metrics_000660 = compute_metrics(df.rename(columns={"000660_종가": "Close"}))

# print(metrics_005930)

annual_return          0.562846
annual_volatility      0.404197
max_drawdown          -0.431663
sharpe_ratio           1.392506
cagr                   0.617825
n_days               483.000000
dtype: float64


In [ ]:
# 1) 과거 주가 가져오는 함수
def fetch_stock_history(ticker: str, years: int = 2) -> pd.DataFrame:
    end = pd.Timestamp.today().strftime("%Y%m%d")
    start = (pd.Timestamp.today() - pd.DateOffset(years=years)).strftime("%Y%m%d")
    df = stock.get_market_ohlcv(start, end, ticker)
    return df

In [ ]:
# 2) 하나의 종목에 대해 compute_metrics 적용하는 래퍼
def compute_stock_metrics_for_ticker(ticker: str, years: int = 2) -> pd.Series:
    df = fetch_stock_history(ticker, years=years)
    # compute_metrics는 Close 컬럼 기준이라고 가정
    metrics = compute_metrics(df.rename(columns={"종가": "Close"}))
    return metrics

In [ ]:
# 3) 두 종목을 한 번에 계산 (decisionAgent에 넘길 payload)
def compute_pair_metrics(t1: str = "005930", t2: str = "000660", years: int = 2):
    m1 = compute_stock_metrics_for_ticker(t1, years=years)
    m2 = compute_stock_metrics_for_ticker(t2, years=years)
    return {
        t1: m1.to_dict(),
        t2: m2.to_dict(),
    }

### Step 0 : 설치와 준비  
Autogen 설치 및 Gemini API 키를 등록하도록 합니다.

In [ ]:
!pip install autogen
!pip install -U "autogen-agentchat"
!pip install "autogen-ext[openai]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.3/119.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 18.6 MB/s eta 0:00:00


In [ ]:
# from google.colab import userdata

# GOOGLE_API_KEY = userdata.get('GEMINI_KEY')

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

### Step 1 : 에이전트 시스템 메세지 작성  
각 에이전트에 적용할 시스템 메세지와 에이전트의 설명을 작성해보겠습니다.  
description 또한 에이전트 선언에 꼭 필요하니, 잘 기억해주세요!  

### Q1. 모델 클라이언트 정의  

#### Hugging Face로 로컬 LLM 클라이언트 만들기

In [ ]:
!pip install -U bitsandbytes accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.9 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "beomi/Llama-3-Open-Ko-8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",          # GPU 자동 할당
)

hf_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    # device 인자는 여기서는 생략해도 됨 (device_map이 처리)
    max_new_tokens=256,         # 일단 토큰 수 줄여서 메모리 여유 확보
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
class LocalHFClient:
    def __init__(self, generator):
        self.generator = generator

    async def generate(self, prompt: str, max_new_tokens: int = 512) -> str:
        # async 인터페이스 맞추기 위해 간단히 래핑
        # (실제론 ThreadPoolExecutor 등을 써서 blocking 피하는 게 베스트)
        outputs = self.generator(
            prompt,
            max_new_tokens=max_new_tokens,
            return_full_text=False,
        )
        return outputs[0]["generated_text"]


In [ ]:
client_small = LocalHFClient(hf_generator)


In [ ]:
# model_client = OpenAIChatCompletionClient(
#     model="gpt-4.1-mini",  # 지금 노트북에서 쓰는 것과 동일하게
#     api_key=OPENAI_API_KEY,
#     base_url="https://api.openai.com/v1",
# )

# client_small = OpenAIChatCompletionClient(
#     model="gpt-4.1-mini",   # 애널리스트용, 빠른 모델
#     api_key=OPENAI_API_KEY,
#     base_url="https://api.openai.com/v1",
# )

# client_decision = OpenAIChatCompletionClient(
#     model="gpt-4.1",        # 최종 의사결정용, 좀 더 큰 모델
#     api_key=OPENAI_API_KEY,
# )

# client_decision = OpenAIChatCompletionClient(
#     model="gpt-4.1-mini",   # 애널리스트용, 빠른 모델
#     api_key=OPENAI_API_KEY,
#     base_url="https://api.openai.com/v1",
# )


### Q2. 에이전트 정의  
위에서 선언한 모델 클라이언트를 사용하는 어시스턴트들을 정의해주세요!  
scenario writer, scene planner, storyboard artist 가 필요합니다  

In [ ]:
# samsung_analyst = AssistantAgent(
#     name="SamsungAnalyst",
#     model_client=client_small,
#     system_message=(
#         "너는 삼성전자(005930) 전문 주식 애널리스트다.\n"
#         "입력으로 주어지는 것은:\n"
#         "1) 오늘 날짜 기준 과거 약 2년치 주가로부터 계산된 정량 지표(metrics)\n"
#         "2) 인터넷에서 수집한 최신 뉴스 및 리포트 요약\n\n"
#         "metrics에는 annual_return, annual_volatility, max_drawdown, sharpe_ratio, cagr, n_days 등이 포함된다.\n"
#         "오직 삼성전자 관점에서만 이 정보를 해석하고, 다음을 한국어로 간결히 작성하라:\n"
#         "- 정량 지표에 대한 평가 (수익률/변동성/샤프/최대낙폭 등)\n"
#         "- 최근 뉴스/이슈가 향후 실적과 밸류에이션에 주는 영향\n"
#         "- 3~6개월 투자 의견 (매수/보유/매도 중 하나, 간단한 근거 포함)\n"
#         "숫자는 그대로 활용하되, 과도한 디스클레이머는 쓰지 마라."
#     ),
# )

# skhynix_analyst = AssistantAgent(
#     name="HynixAnalyst",
#     model_client=client_small,
#     system_message=(
#         "너는 SK하이닉스(000660) 전문 주식 애널리스트다.\n"
#         "입력으로 주어지는 것은:\n"
#         "1) 오늘 날짜 기준 과거 약 2년치 주가로부터 계산된 정량 지표(metrics)\n"
#         "2) 인터넷에서 수집한 최신 뉴스 및 리포트 요약\n\n"
#         "metrics에는 annual_return, annual_volatility, max_drawdown, sharpe_ratio, cagr, n_days 등이 포함된다.\n"
#         "오직 SK하이닉스 관점에서만 이 정보를 해석하고, 다음을 한국어로 간결히 작성하라:\n"
#         "- 정량 지표에 대한 평가\n"
#         "- 최근 뉴스/이슈가 향후 실적과 밸류에이션에 주는 영향\n"
#         "- 3~6개월 투자 의견 (매수/보유/매도 중 하나, 간단한 근거 포함)"
#     ),
# )

SAMSUNG_ANALYST_PROMPT = """
너는 삼성전자(005930) 전문 주식 애널리스트다.
입력으로 주어지는 것은:
1) 오늘 날짜 기준 과거 약 2년치 주가로부터 계산된 정량 지표(metrics)
2) 인터넷에서 수집한 최신 뉴스 및 리포트 요약

metrics에는 annual_return, annual_volatility, max_drawdown, sharpe_ratio, cagr, n_days 등이 포함된다.
오직 삼성전자 관점에서만 이 정보를 해석하고, 다음을 한국어로 간결히 작성하라:
- 정량 지표에 대한 평가 (수익률/변동성/샤프/최대낙폭 등)
- 최근 뉴스/이슈가 향후 실적과 밸류에이션에 주는 영향
- 3~6개월 투자 의견 (매수/보유/매도 중 하나, 간단한 근거 포함)
숫자는 그대로 활용하되, 과도한 디스클레이머는 쓰지 마라.
""".strip()

HYNYX_ANALYST_PROMPT = """
너는 SK하이닉스(000660) 전문 주식 애널리스트다.
입력으로 주어지는 것은:
1) 오늘 날짜 기준 과거 약 2년치 주가로부터 계산된 정량 지표(metrics)
2) 인터넷에서 수집한 최신 뉴스 및 리포트 요약

metrics에는 annual_return, annual_volatility, max_drawdown, sharpe_ratio, cagr, n_days 등이 포함된다.
오직 SK하이닉스 관점에서만 이 정보를 해석하고, 다음을 한국어로 간결히 작성하라:
- 정량 지표에 대한 평가
- 최근 뉴스/이슈가 향후 실적과 밸류에이션에 주는 영향
- 3~6개월 투자 의견 (매수/보유/매도 중 하나, 간단한 근거 포함)
""".strip()


In [ ]:
async def run_samsung_analyst(prompt_body: str) -> str:
    # prompt_body: build_analyst_prompt_from_news_report가 만든 텍스트
    full_prompt = f"{SAMSUNG_ANALYST_PROMPT}\n\n[입력]\n{prompt_body}"
    return await client_small.generate(full_prompt, max_new_tokens=512)

async def run_hynix_analyst(prompt_body: str) -> str:
    full_prompt = f"{HYNYX_ANALYST_PROMPT}\n\n[입력]\n{prompt_body}"
    return await client_small.generate(full_prompt, max_new_tokens=512)


In [ ]:
# news_agent = AssistantAgent(
#     name="NewsAgent",
#     model_client=client_small,
#     system_message=(
#         "너는 한국 주식 뉴스 요약 애널리스트다.\n"
#         "입력으로 티커(예: 005930)와 회사명, 그리고 검색 결과 텍스트 목록이 주어진다.\n"
#         "검색 결과를 바탕으로 해당 종목에 대한 최근 1~3개월 주요 이슈를 간결히 요약하라.\n"
#         "요약 형식:\n"
#         "- 핵심 이슈 3~5개 (불릿)\n"
#         "- 전반적인 톤 (긍정/중립/부정) 한 줄\n"
#         "반드시 한국어로 작성하고, 불필요한 각주나 출처 표기는 하지 마라."
#     ),
# )

NEWS_SYSTEM_PROMPT = """
너는 한국 주식 뉴스 요약 애널리스트다.
입력으로 티커(예: 005930)와 회사명, 그리고 검색 결과 텍스트 목록이 주어진다.
검색 결과를 바탕으로 해당 종목에 대한 최근 1~3개월 주요 이슈를 간결히 요약하라.

요약 형식:
- 핵심 이슈 3~5개 (불릿)
- 전반적인 톤 (긍정/중립/부정) 한 줄

반드시 한국어로 작성하고, 불필요한 각주나 출처 표기는 하지 마라.
""".strip()

In [ ]:
# decision_agent = AssistantAgent(
#     name="DecisionAgent",
#     model_client=client_decision,
#     system_message=(
#         "너는 포트폴리오 매니저다.\n"
#         "삼성전자(005930)와 SK하이닉스(000660)의 정량 지표(metrics)와 "
#         "각 종목 애널리스트의 리포트를 보고, 두 종목 중 어느 쪽이 더 매력적인지 판단하는 역할이다.\n\n"
#         "출력 요구사항:\n"
#         "1) 정량 지표 비교 요약 (표 1개: 행=지표, 열=005930/000660)\n"
#         "2) 각 애널리스트 리포트의 핵심 포인트 2~3개씩\n"
#         "3) 오늘 기준 3~6개월 관점에서의 최종 종합 의견:\n"
#         "   - 삼성전자 vs 하이닉스 중 상대적으로 더 선호하는 종목과 이유\n"
#         "   - 두 종목 모두에 대한 간단한 리스크 요인 언급\n"
#         "4) 반드시 한국어로 작성.\n"
#     ),
# )

DECISION_SYSTEM_PROMPT = """
너는 포트폴리오 매니저다.
삼성전자(005930)와 SK하이닉스(000660)의 정량 지표(metrics)와
각 종목 애널리스트의 리포트를 보고, 두 종목 중 어느 쪽이 더 매력적인지 판단하는 역할이다.

출력 요구사항:
1) 정량 지표 비교 요약 (표 1개: 행=지표, 열=005930/000660)
2) 각 애널리스트 리포트의 핵심 포인트 2~3개씩
3) 오늘 기준 3~6개월 관점에서의 최종 종합 의견:
   - 삼성전자 vs 하이닉스 중 상대적으로 더 선호하는 종목과 이유
   - 두 종목 모두에 대한 간단한 리스크 요인 언급
4) 반드시 한국어로 작성.
""".strip()

### 뉴스 검색

In [ ]:
!pip install gnews

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 29.2 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=202e378361936a1cd51391bb7469ab6b3f4857bf5c9cd705f70158d55eec79dc
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


In [ ]:
from gnews import GNews

In [ ]:
# gnews = GNews(language='ko', country='KR', max_results=5, period='7d')

# def search_news_raw(ticker: str, company_name: str, n_results: int = 5) -> list[str]:
#     query = f"{company_name} {ticker}"
#     articles = gnews.get_news(query)  # 최근 7일 한국어 뉴스 검색[web:75]

#     texts: list[str] = []
#     for a in articles[:n_results]:
#         # title + description을 합쳐서 에이전트에 넘길 텍스트로 사용
#         title = a.get("title") or ""
#         desc = a.get("description") or ""
#         text = (title + " " + desc).strip()
#         if text:
#             texts.append(text)

#     return texts

In [ ]:
# def build_news_agent_prompt(ticker: str, company_name: str, raw_texts: list[str]) -> str:
#     news_text = "\n\n".join(raw_texts) if raw_texts else "관련 뉴스가 거의 없음."

#     return (
#         f"티커: {ticker}\n"
#         f"회사명: {company_name}\n\n"
#         "다음은 인터넷/뉴스 검색 결과 텍스트 목록이다:\n"
#         f"{news_text}\n\n"
#         "이 정보를 바탕으로 지시에 따라 요약하라."
#     )


In [ ]:
async def get_news_report_from_manual_input(
    ticker: str,
    company_name: str,
    news_list: list[str],
) -> str:
    joined_texts = "\n\n".join(news_list)

    prompt = f"""{NEWS_SYSTEM_PROMPT}

티커: {ticker}
회사명: {company_name}

[사용자가 정리한 최근 뉴스]
{joined_texts}
"""

    report = await client_small.generate(
        prompt,
        max_new_tokens=160,  # 128~256 정도로 유지
    )
    return report

In [ ]:
# import asyncio

# async def get_news_report_from_agent(ticker: str, company_name: str) -> str:
#     raw_texts = search_news_raw(ticker, company_name, n_results=5)

#     # prompt = build_news_agent_prompt(ticker, company_name, raw_texts)
#     # # result = await news_agent.run(task=prompt)
#     # result = await client_small.generate(prompt, max_new_tokens=512)
#     # return result

#     joined_texts = "\n\n".join(raw_texts[:3])  # 상위 3개만 사용

#     prompt = f"""{NEWS_SYSTEM_PROMPT}

#     티커: {ticker}
#     회사명: {company_name}

#     [검색 결과 텍스트]
#     {joined_texts}
#     """

#     report = await client_small.generate(
#         max_new_tokens=160,   # 128~256 사이로 줄이기
#         temperature=0.7,
#         top_p=0.9,
#     )

#     return report


# # # 사용 예시
# # result = asyncio.run(get_news_report_from_agent("005930", "삼성전자"))


In [ ]:
def build_analyst_prompt_from_news_report(
    ticker: str,
    metrics: dict,
    news_report: str,
) -> str:
    return (
        f"티커: {ticker}\n"
        f"정량 지표(metrics):\n{metrics}\n\n"
        "다음은 NewsAgent가 작성한 최근 1~3개월 뉴스 요약 보고서이다:\n"
        f"{news_report}\n\n"
        "이 정량 지표와 뉴스 요약을 모두 반영해서 분석하라."
    )

#### 뉴스 캐시 유틸 함수 추가

In [ ]:
# import os
# import json

# CACHE_DIR = "news_cache"
# os.makedirs(CACHE_DIR, exist_ok=True)

# def load_news_cache(ticker: str) -> str | None:
#     path = os.path.join(CACHE_DIR, f"{ticker}.json")
#     if not os.path.exists(path):
#         return None
#     try:
#         with open(path, "r", encoding="utf-8") as f:
#             data = json.load(f)
#         return data.get("report")
#     except Exception:
#         return None

# def save_news_cache(ticker: str, report: str) -> None:
#     path = os.path.join(CACHE_DIR, f"{ticker}.json")
#     with open(path, "w", encoding="utf-8") as f:
#         json.dump({"ticker": ticker, "report": report}, f, ensure_ascii=False, indent=2)


In [ ]:
# async def get_news_report_with_cache(ticker: str, company_name: str) -> str:
#     # 1) 캐시 체크
#     cached = load_news_cache(ticker)
#     if cached is not None:
#         return cached

#     # 2) 캐시 없으면 NewsAgent 호출
#     report = await get_news_report_from_agent(ticker, company_name)

#     # 3) 결과 저장
#     save_news_cache(ticker, report)
#     return report


### Q3. 그룹챗 선언  
그룹쳇에는 위에 선언한 세 개의 에이전트가 들어가야 합니다!  
챗 종료를 관리하기 위한 termination을 정의하고 시작해보죠

In [ ]:
# from autogen_agentchat.teams import RoundRobinGroupChat

In [ ]:
# termination = TextMentionTermination("TERMINATE")

# groupchat = RoundRobinGroupChat(
#     [samsung_analyst, skhynix_analyst, decision_agent],
#     termination_condition=termination,
# )

In [ ]:
# 여러 번 그룹챗을 수행할 경우, 꼭 위의 그룹챗 선언 전에 해당 문구를 먼저 실행해주시기 바랍니다
# await group_chat.reset()

### Q4. 스트림 실행하기  
콘솔에서 아래 선언한 스트림을 실행해보세요!  
결과를 result 변수로 받아 추후 작업을 수행할 수 있습니다.

In [ ]:
# async def run_full_pipeline_with_news_agent(years: int = 2):
#     # 1) 정량 지표 계산
#     metrics_dict = compute_pair_metrics("005930", "000660", years=years)
#     metrics_005930 = metrics_dict["005930"]
#     metrics_000660 = metrics_dict["000660"]

#     # 2) NewsAgent가 각 종목 뉴스 요약 보고서 작성
#     samsung_news_report = await get_news_report_with_cache("005930", "삼성전자")
#     await asyncio.sleep(1.5)
#     hynix_news_report   = await get_news_report_with_cache("000660", "SK하이닉스")
#     await asyncio.sleep(1.5)

#     # 3) 애널리스트 프롬프트 구성 (정량 + NewsAgent 보고서)
#     samsung_prompt = build_analyst_prompt_from_news_report(
#         "005930",
#         metrics_005930,
#         samsung_news_report,
#     )
#     hynix_prompt = build_analyst_prompt_from_news_report(
#         "000660",
#         metrics_000660,
#         hynix_news_report,
#     )

#     samsung_report = await samsung_analyst.run(task=samsung_prompt)
#     await asyncio.sleep(1.5)
#     hynix_report   = await skhynix_analyst.run(task=hynix_prompt)
#     await asyncio.sleep(1.5)

#     # 4) DecisionAgent에 모든 정보 전달
#     decision_prompt = (
#         "다음은 삼성전자/하이닉스의 백테스트 지표, NewsAgent 뉴스 요약, "
#         "각 종목 애널리스트 리포트이다.\n\n"
#         f"[삼성전자(005930) metrics]\n{metrics_005930}\n\n"
#         f"[하이닉스(000660) metrics]\n{metrics_000660}\n\n"
#         "[삼성전자 뉴스 요약 (NewsAgent)]\n"
#         f"{samsung_news_report}\n\n"
#         "[하이닉스 뉴스 요약 (NewsAgent)]\n"
#         f"{hynix_news_report}\n\n"
#         "[삼성전자 애널리스트 리포트]\n"
#         f"{samsung_report}\n\n"
#         "[하이닉스 애널리스트 리포트]\n"
#         f"{hynix_report}\n\n"
#         "위 정보를 바탕으로 요구된 형식대로 최종 결론을 작성하라."
#     )

#     final_decision = await decision_agent.run(decision_prompt)

#     return {
#         "metrics": metrics_dict,
#         "samsung_news_report": samsung_news_report,  # ← NewsAgent 산출물
#         "hynix_news_report": hynix_news_report,      # ← NewsAgent 산출물
#         "samsung_report": samsung_report,            # ← 삼성 애널리스트
#         "hynix_report": hynix_report,                # ← 하이닉스 애널리스트
#         "final_decision": final_decision,            # ← DecisionAgent
#     }

In [ ]:
async def run_decision_agent(
    metrics_dict,
    samsung_news_report: str,
    hynix_news_report: str,
    samsung_report: str,
    hynix_report: str,
) -> str:
    metrics_005930 = metrics_dict["005930"]
    metrics_000660 = metrics_dict["000660"]

    decision_prompt = f"""{DECISION_SYSTEM_PROMPT}

[삼성전자(005930) metrics]
{metrics_005930}

[하이닉스(000660) metrics]
{metrics_000660}

[삼성전자 뉴스 요약 (NewsAgent)]
{samsung_news_report}

[하이닉스 뉴스 요약 (NewsAgent)]
{hynix_news_report}

[삼성전자 애널리스트 리포트]
{samsung_report}

[하이닉스 애널리스트 리포트]
{hynix_report}
"""

    # client_decision 은 HuggingFace/로컬 LLM 래퍼라고 가정
    return await client_small.generate(decision_prompt, max_new_tokens=1024)

In [ ]:
samsung_news_list = [
    "삼성전자, HBM3E 대량 양산 시작… 엔비디아향 공급 확대 기대",
    "삼성전자, 2024년 주주환원 정책 발표… 배당 확대 가능성 부각",
    "글로벌 반도체 업황 개선 기대감에 외국인 매수세 유입",
]

hynix_news_list = [
    "SK하이닉스, HBM3E 대량 양산 시작… 엔비디아향 공급 확대 기대",
    "SK하이닉스, 2024년 주주환원 정책 발표… 배당 확대 가능성 부각",
    "글로벌 반도체 업황 개선 기대감에 외국인 매수세 유입",
]


In [ ]:
import time

async def run_full_pipeline_with_news_agent(years: int = 2):
    t0 = time.time()

    # 1) 정량 지표 계산
    print("start metrics", flush=True)
    metrics_dict = compute_pair_metrics("005930", "000660", years=years)
    metrics_005930 = metrics_dict["005930"]
    metrics_000660 = metrics_dict["000660"]
    print("done metrics", time.time() - t0, flush=True)

    # 2) NewsAgent가 각 종목 뉴스 요약 보고서 작성
    print("start samsung news", flush=True)
    # samsung_news_report = await get_news_report_with_cache("005930", "삼성전자")
    samsung_news_report = await get_news_report_from_manual_input("005930", "삼성전자", samsung_news_list)
    print("done samsung news", time.time() - t0, flush=True)
    # await asyncio.sleep(1.5)

    print("start hynix news", flush=True)
    # hynix_news_report   = await get_news_report_with_cache("000660", "SK하이닉스")
    hynix_news_report   = await get_news_report_from_manual_input("000660", "SK하이닉스", hynix_news_list)
    print("done hynix news", time.time() - t0, flush=True)
    # await asyncio.sleep(1.5)

    # 3) 애널리스트 프롬프트 구성 (정량 + NewsAgent 보고서)
    samsung_prompt = build_analyst_prompt_from_news_report(
        "005930",
        metrics_005930,
        samsung_news_report,
    )
    hynix_prompt = build_analyst_prompt_from_news_report(
        "000660",
        metrics_000660,
        hynix_news_report,
    )

    print("start samsung analyst", flush=True)
    samsung_report = await run_samsung_analyst(samsung_prompt)
    print("done samsung analyst", time.time() - t0, flush=True)
    # await asyncio.sleep(1.5)

    print("start hynix analyst", flush=True)
    hynix_report   = await run_hynix_analyst(hynix_prompt)
    print("done hynix analyst", time.time() - t0, flush=True)
    # await asyncio.sleep(1.5)

    # 4) DecisionAgent에 모든 정보 전달

    print("start decision", flush=True)
    final_decision = await run_decision_agent(
        metrics_dict, samsung_news_report, hynix_news_report,
        samsung_report, hynix_report,
    )
    print("done decision", time.time() - t0, flush=True)

    return {
        "metrics": metrics_dict,
        "samsung_news_report": samsung_news_report,
        "hynix_news_report": hynix_news_report,
        "samsung_report": samsung_report,
        "hynix_report": hynix_report,
        "final_decision": final_decision,
    }

In [ ]:
result = await run_full_pipeline_with_news_agent(years=2)

print("=== SAMSUNG NEWS (NewsAgent) ===")
print(result["samsung_news_report"])

print("\n=== HYNIX NEWS (NewsAgent) ===")
print(result["hynix_news_report"])

print("\n=== SAMSUNG ANALYST REPORT ===")
print(result["samsung_report"])

print("\n=== HYNIX ANALYST REPORT ===")
print(result["hynix_report"])

print("\n=== FINAL DECISION ===")
print(result["final_decision"])

start metrics
done metrics 0.9996776580810547
start samsung news


Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done samsung news 153.9642903804779
start hynix news


Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done hynix news 297.0033061504364
start samsung analyst


...

done samsung analyst 750.1792161464691
start hynix analyst


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done hynix analyst 1203.4719495773315
start decision


Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done decision 2137.355451107025
=== SAMSUNG NEWS (NewsAgent) ===
[기계학습 모델이 정리한 주요 뉴스]
삼성전자, HBM3E 대량 양산 시작… 엔비디아향 공급 확대 기대 삼성전자가 2분기부터 차세대 HBM3E 제품 양산을 시작했다. 삼성전자는 3세대 HBM(High Bandwidth Memory)인 HBM3E를 2분기부터 양산하고 있다. HBM3E는 메모리 반도체의 일종인 D램으로, 대용량 데이터를 초고속으로 처리할 수 있어 인공지능(AI)이나 자율주행 등 4차 산업혁명에 필수적인 반도체다. 삼성전자는 이달 초 미국에서 열린 글

=== HYNIX NEWS (NewsAgent) ===
SK하이닉스, HBM3E 대량 양산 시작… 엔비디아향 공급 확대 기대
SK하이닉스가 차세대 서버용 메모리 제품인 HBM3E 양산을 시작했다. HBM3E는 SK하이닉스의 자체 개발한 '초고적층(Quad-Stack)' 공정 기술을 적용해 3세대(1y) 제품과 비교해 생산성은 40%, 성능은 1.5배 향상됐다. HBM3E는 차세대 인공지능(AI) 컴퓨팅에 최적화된 제품으로 서버용 고성능 메모리 시장에서 높은 수요가 예상된다. SK하이닉스는 엔비

=== SAMSUNG ANALYST REPORT ===
이들은 “민주당이 추진하는 미디어법은 한마디로 말해 신문, 방송, 통신의 지배구조를 바꿔 언론의 독립성과 다양성을 보장하고 언론의 자유를 회복하는 것”이라며 “민주당은 미디어법을 통해 대기업의 방송 진입을 제한하고, 신문과 방송의 겸영을 금지하겠다는 것”이라고 설명했다.이거 보고 보물섬 구독함김(이름) 님은 안웃기고 재미없어요또한, 공기조화기와 같은 실내기에 구비되는 인버터 압축기의 경우, 제어부가 구비되며, 상기 제어부는, 상기 실내기의 운전 조건에 따라 인버터 압축기의 구동을 제어한다.
이런 이유로 신종 코로나바이러스가 전파될 수 있는 장소는 무한대다. 감염자를 접촉한 모든 사람이 감염될 수 있기 때문이다. 감염자가 기침을 하면 기침에